In [ ]:
from google.colab import files
files.upload()

Saving crops1.txt to crops1.txt


{'crops1.txt': b'Rice,Kharif,10,800,120,4500 \r\nWheat,Rabi,8,300,90,2800\r\nMaize,Kharif,6,600,100,3200 \r\nCotton,Kharif,12,700,140,3600 \r\nSugarcane,Annual,15,1200,200,9000 \r\nPulses,Rabi,5,250,60,1100 \r\nGroundnut,Kharif,7,500,80,1800 Soybean,Kharif,9,650,110,2700 \r\nMustard,Rabi,6,300,70,1500 \r\nBarley,Rabi,7,280,65,1600 \r\nSorghum,Kharif,8,550,90,2400 \r\nMillet,Kharif,5,400,60,1400 Sunflower,Kharif,6,450,85,1700 Potato,Rabi,4,350,120,3000 \r\nOnion,Rabi,5,320,100,2800 \r\nTomato,Kharif,4,500,110,2600 \r\nChickpea,Rabi,6,260,70,1400 \r\nPeanut,Kharif,7,480,85,1900 \r\nMaize,Rabi,6,320,95,2900 \r\nRice,Rabi,9,450,130,4200'}

In [ ]:
#Agricultural Data Analysis Project
#-------Function Definition------
def load_data(filename):
  """Reads crop data from a file and returns a list of dictionaries"""
  crops=[]
  file=open(filename,"r")
  for line in file:
      line=line.strip()
      if not line:
          continue

      # The problematic line: Groundnut,Kharif,7,500,80,1800 Soybean,Kharif,9,650,110,2700
      # It contains '1800 ' followed by 'Soybean'. This indicates a missing comma.
      # Replace '1800 Soybean' with '1800,Soybean' to correctly split.
      # This fix addresses the specific malformation observed in the input file.
      if "1800 Soybean" in line:
          line = line.replace("1800 Soybean", "1800,Soybean")

      # Now process the (potentially modified) line
      data_entries = line.split(",")

      num_expected_fields = 6 # Each crop record is expected to have 6 fields

      # Iterate through the split data_entries to extract individual crop records
      # This handles cases where multiple records might be on a single line after replacement.
      i = 0
      while i < len(data_entries):
          if i + num_expected_fields <= len(data_entries):
              record_data = data_entries[i : i + num_expected_fields]

              if len(record_data) == num_expected_fields:
                  try:
                      crop={
                          "name": record_data[0].strip(),
                          "season": record_data[1].strip(),
                          "area": float(record_data[2].strip()),
                          "rainfall": float(record_data[3].strip()),
                          "fertiliser": float(record_data[4].strip()),
                          "yield": float(record_data[5].strip())
                      }
                      crops.append(crop)
                  except ValueError as e:
                      print(f"Error parsing record {record_data} on line '{line}': {e}")
                  except IndexError as e:
                      print(f"Index error for record {record_data} on line '{line}': {e}")
              else:
                  print(f"Warning: Skipping partial record {record_data} on line '{line}'")
              i += num_expected_fields
          else:
              print(f"Warning: Skipping remaining malformed data on line '{line}': {data_entries[i:]}")
              break # Exit if remaining parts don't form a full record
  file.close()
  return crops

def calculate_yield_per_area(crop):
  """Return yield per unit area"""
  return crop["yield"]/crop["area"]

def display_crops(crops):
  """Display all crop records"""
  print("\nCrop Records:")
  for crop in crops:
    ypa=calculate_yield_per_area(crop)
    print(crop["name"],"-",crop["season"],"-Yield/Area:",round(ypa,2))

def find_highest_and_lowest_yield(crops):
 """Find crops with highest and lowest yield per area"""
 # Ensure crops list is not empty before accessing elements
 if not crops:
    print("No crops data to analyze for highest/lowest yield.")
    return None, None
 highest=crops[0]
 lowest=crops[0]
 for crop in crops:
  if calculate_yield_per_area(crop)>calculate_yield_per_area(highest):
    highest=crop
  if calculate_yield_per_area(crop)<calculate_yield_per_area(lowest):
    lowest=crop
 return highest,lowest

def seasonal_summary(crops):
  """Generate seasonal yield summary using dictionary"""
  summary={}

  for crop in crops:
    season=crop["season"]
    ypa=calculate_yield_per_area(crop)

    if season not in summary:
      summary[season]=[]
    summary[season].append(ypa)
  return summary

def crop_count_per_season(crops):
  count={}

  for crop in crops:
    season=crop["season"]
    count[season] = count.get(season, 0) + 1 # Corrected counting logic
  return count # Return the dictionary

def fertilizer_efficiency(crop):
  return crop["yield"]/crop["fertiliser"]

def most_fertilizer_efficient_crop(crops):
   best= crops[0]

   for crop in crops:
     if fertilizer_efficiency(crop)>fertilizer_efficiency(best):
      best=crop
   print("Most Fertilizer Efficient Crop:",best["name"])

def

def write_report(filename,crops,highest,lowest,summary, crop_counts):
  """Write analysis results to output file"""
  file= open(filename,"w")
  file.write("Agricultural Data Analysis Report\n\n")

  file.write("Crop Yield per Unit Area:\n")
  for crop in crops:
        ypa=calculate_yield_per_area(crop)
        file.write(crop["name"]+":"+str(round(ypa,2))+"\n")

  if highest:
      file.write("\nHighest Yield Crop:\n")
      file.write(highest["name"]+"\n")

  if lowest:
      file.write("\nlowest Yield Crop:\n")
      file.write(lowest["name"]+"\n")

  file.write("\nSeasonal Average Yield:\n")
  for season in summary:
      if summary[season]: # Ensure there are values before calculating average
          avg=sum(summary[season])/len(summary[season])
          file.write(season +":"+str(round(avg,2))+"\n") # Fixed: Moved file.write inside the loop
      else:
          file.write(season + ": No data\n")

  file.write("\nCrop count per season:\n") # Corrected placement of this line
  for season,value in crop_counts.items(): # Use the passed crop_counts dictionary
      file.write(season + ":" + str(value) + "\n")

  file.close() # Close the file after all writing is done

#------Main Program------
def main():
  input_file="crops1.txt"
  output_file="analysis_report.txt"

  crops=load_data(input_file)

  # Check if crops data was loaded successfully before proceeding
  if not crops:
      print("No crop data loaded. Exiting.")
      return

  for crop in crops:
    print(crop)

  for crop in crops:
     for key,value in crop.items():
      print(f"{key}:{value}")

  print("-----")

  display_crops(crops)

  highest,lowest=find_highest_and_lowest_yield(crops)

  summary=seasonal_summary(crops)

  crop_counts = crop_count_per_season(crops)
  write_report(output_file,crops,highest,lowest,summary, crop_counts)

  print("\nAnalysis complete.Report written to", output_file)

main()

Error parsing record ['Millet', 'Kharif', '5', '400', '60', '1400 Sunflower'] on line 'Millet,Kharif,5,400,60,1400 Sunflower,Kharif,6,450,85,1700 Potato,Rabi,4,350,120,3000': could not convert string to float: '1400 Sunflower'
Error parsing record ['Kharif', '6', '450', '85', '1700 Potato', 'Rabi'] on line 'Millet,Kharif,5,400,60,1400 Sunflower,Kharif,6,450,85,1700 Potato,Rabi,4,350,120,3000': could not convert string to float: '1700 Potato'
{'name': 'Rice', 'season': 'Kharif', 'area': 10.0, 'rainfall': 800.0, 'fertiliser': 120.0, 'yield': 4500.0}
{'name': 'Wheat', 'season': 'Rabi', 'area': 8.0, 'rainfall': 300.0, 'fertiliser': 90.0, 'yield': 2800.0}
{'name': 'Maize', 'season': 'Kharif', 'area': 6.0, 'rainfall': 600.0, 'fertiliser': 100.0, 'yield': 3200.0}
{'name': 'Cotton', 'season': 'Kharif', 'area': 12.0, 'rainfall': 700.0, 'fertiliser': 140.0, 'yield': 3600.0}
{'name': 'Sugarcane', 'season': 'Annual', 'area': 15.0, 'rainfall': 1200.0, 'fertiliser': 200.0, 'yield': 9000.0}
{'name': 